# Problem Set 5: Can You Predict Party from State of the Union Addresses?

QSS 20 · Modern Statistical Computing · Fall 2026

**Out:** Thursday, October 29 · **Due:** Thursday, November 5, 11:59pm

## Overview

Problem Set 4 ended with a number that looks like a finding. The word
`japanese` appears about 28 times more often in Democratic than in Republican
State of the Union addresses. If single words separate the parties that
sharply, a classifier trained on all of them should separate the addresses
easily. However, it does not. A model that reports 67 percent accuracy on a
two-class problem can be worthless, better than chance, or actively misleading.

This problem set fits the three classifiers of lecture 13 to the same
addresses: logistic regression, k-nearest neighbors, and a decision tree. It
reads each one the way the lecture asked: against the baseline, on rows the fit
never saw, and with the precision and recall of the party in question. The last
section drops the labels and asks whether k-means finds the parties on its own.

You do not need Problem Set 4 to do this one. The corpus is supplied, already
parsed and merged.

**Submission.** Work in `ps5.ipynb`. Each task has an empty cell under it and
a page break after it. Keep that layout, so page tagging on Gradescope stays
simple. Run Kernel, then Restart Kernel and Run All Cells. Export with
`quarto render ps5.ipynb --to typst`. Upload `ps5.ipynb` and `ps5.pdf` to
Gradescope. The Problem Set Reflections at the end are part of the set.

**Data.** `data/speeches.csv` holds 96 rows, one per State of the Union address
from 1934 to 2026, with columns `year`, `president`, `party`, `decade`,
`length`, and `text`. The row unit is one address, not one president: Franklin
Roosevelt contributes twelve rows. The table is the cleaned output of Problem
Set 4's Section 1, restricted to the modern era and supplied here so that your
work on this problem set does not depend on your work on that one.

**Tools.** Every call you need appeared in lectures 11 to 14: `.str.findall`,
`TfidfVectorizer`, `train_test_split`, `LogisticRegression`,
`KNeighborsClassifier`, `DecisionTreeClassifier`, `plot_tree`,
`confusion_matrix`, `classification_report`, `KMeans`, and `PCA`.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SPEECHES = Path("data/speeches.csv")

{{< pagebreak >}}

## 1. Load the corpus and check the claim (7 points)

**1.1 (7 points).** Read the table. Report the number of rows, the rows per
party, and the number of distinct presidents. Then count the uses of
`japanese` in each address with `.str.findall` and `.str.len`, as in Problem
Set 4, and print the five addresses that use it most, with year, president,
and party. Which years do the five come from, and does that make `japanese` a
party word or a period word?

{{< pagebreak >}}

## 2. State the baseline before you model (4 points)

**2.1 (4 points).** Compute the accuracy of a rule that always guesses the
more common outcome variable (party) in your data. Report it, and say why a classifier
must beat this baseline before we can claim it has learned anything about
party.

{{< pagebreak >}}

## 3. Logistic regression (16 points)

**3.1 (5 points).** Build the TF-IDF matrix of all 96 addresses with
`stop_words="english"` and `min_df=2`, and hold it as a DataFrame named
`tfidf`, with the words as columns and the index of `speeches`, as in lecture
13. Report its shape.

{{< pagebreak >}}

**3.2 (6 points).** Split the rows of `tfidf` and the party with
`train_test_split`, holding out a quarter of them, with `random_state=42` and
`stratify=` the party. Fit `LogisticRegression(max_iter=2000)` on the training
rows and report its accuracy on the test rows **and** on the training rows. Say
in one sentence what the gap between the two numbers means.

{{< pagebreak >}}

**3.3 (5 points).** Produce the confusion matrix on the test rows, with
labeled rows and columns, Democratic first. Which party does the model get
wrong, and in which direction?

{{< pagebreak >}}

## 4. Accuracy, precision, and recall (17 points)

Lecture 13 read a classifier as a net: the positive class is the fish, and the
confusion matrix counts what the net caught and what it missed. The matrix in
3.3 shows that one accuracy number hides which party the model finds. In this
section the positive class is Republican, the smaller party, as Madison was in
the lecture.

**4.1 (7 points).** Take the four cells of the confusion matrix from 3.3 with
Republican as the positive class. Give the counts of true positives, false
positives, false negatives, and true negatives. Compute accuracy, precision,
and recall by hand (without using functions but just plain arithmetic in python) from those four counts, each as a fraction and as a
decimal. Then confirm all three with `classification_report`.

{{< pagebreak >}}

**4.2 (5 points).** Now take the rule from Section 2 that answers Democratic
for every address. Without fitting anything, write down its confusion matrix
on the 24 test rows, then its accuracy, precision, and recall with Republican
as the positive class, and its precision and recall with Democratic as the
positive class. Check your numbers with `classification_report` on a Series
of 24 `"Democratic"` labels, passing `zero_division=0`. Which positive class
gives this rule (always predicting Democrat) a recall of 1.0, and what does that show about reading recall
on its own?

{{< pagebreak >}}

**4.3 (5 points).** Two readers of this classifier. A historian wants to read
every Republican address that mentions inflation and does not mind skimming a
few Democratic ones by mistake. An editor wants to quote three addresses as
examples of Republican rhetoric and must not attribute a Democrat's words to
the other party. Name the measure, precision or recall, that each reader
needs most, with one reason each.

*Your answer:*

{{< pagebreak >}}

## 5. k-nearest neighbors (15 points)

Logistic regression fits one weight per word. k-nearest neighbors fits
nothing: for a new address it finds the k training addresses whose rows of
`tfidf` are closest, by the ordinary distance across all 9,908 columns, and
answers the majority party among them. Use the same split as 3.2 and the
columns of `tfidf` as they are.

**5.1 (6 points).** Fit `KNeighborsClassifier(n_neighbors=5)` on the training
rows. Report its training and test accuracy and its confusion matrix on the
test rows, labeled as in 3.3, then Republican's precision and recall from
`classification_report`. How many of the 11 Republican test addresses does it
catch, against the 3 that logistic regression caught in 3.3?

{{< pagebreak >}}

**5.2 (5 points).** The number of neighbors is a choice. For each k in
`[1, 3, 5, 9, 15, 25, 45]`, report the training accuracy, the test accuracy, in
one table with k as the index. Then, in two sentences: what happens to the
two numbers as k grows, and what does the model answer for every address at
k = 72, the number of training rows? Check the last claim by fitting with
`n_neighbors=72` and scoring the test rows.

{{< pagebreak >}}

**5.3 (4 points).** The 2026 address is among the test rows. Use
`kneighbors` on its row of `tfidf`, as lecture 13 did for essay 49, to list
its five nearest training addresses with year, president, party, and
distance. Who delivered the five?

{{< pagebreak >}}

## 6. Decision trees (15 points)

A decision tree asks yes-or-no questions about one word's weight at a time:
is the weight of some word at most some cutoff? With 9,908 columns, the fit
chooses the word and the cutoff at each question itself, which makes the tree
the one classifier here that names the words it uses.

**6.1 (7 points).** Fit `DecisionTreeClassifier(max_depth=2, random_state=0)`
on the training rows and draw it with `plot_tree`, passing
`feature_names=list(tfidf.columns)`, `class_names=tree.classes_`,
`filled=True`, and `impurity=False`. Report its training and test accuracy.
Name the word at the root and the two words at the second level, and the
label the tree gives an address that goes left at the root and right at the
second question.
Then print the five addresses with the largest weight on the root word, with
year, president, and party. Which party and which decades do the five come
from?

{{< pagebreak >}}

**6.2 (8 points).** Loop over `max_depth` from 1 to 6. For each depth, report
the number of leaves (`get_n_leaves`), the training accuracy, and the test
accuracy, in one table. Then put the three classifiers side by side, logistic
regression from 3.2, k-nearest neighbors with k = 5 from 5.1, and the tree of
depth 2 from 6.1, in a table with test accuracy, Republican precision, and
Republican recall, computed from the cells of each confusion matrix as in
4.1. Which classifier has the highest test accuracy?

{{< pagebreak >}}

## 7. K-means clustering (16 points)

None of the three classifiers heard the party clearly. k-means asks the
reverse question: shown no labels at all, does the algorithm split the 96
addresses into two groups that match the parties, or does it find some other
division?

**7.1 (8 points).** Fit `KMeans(n_clusters=2, n_init=1000, random_state=42)`
on `tfidf` with `fit_predict`, as in lecture 14, and store the labels in a
new column `speeches["cluster"]`. Then make two tables with `pd.crosstab`:
party against cluster, and decade against cluster. Answer three questions
from the tables: how many Democratic and how many Republican addresses fall
in each cluster; which decades fall entirely in one cluster; and, printing
the addresses of the 1980s with year, president, and cluster, in which year
the cluster label changes. Hint: `n_init=1000` runs k-means from 1,000
random starts and keeps the best.

{{< pagebreak >}}

**7.2 (8 points).** Reduce `tfidf` to two components with
`PCA(n_components=2, random_state=0)`. Plot the addresses on the two
components, colored by party, with each point labeled by its year
(`ax.annotate` with `fontsize=6`). Then answer from the plot: which end of component 1 holds the oldest
addresses and which the newest; whether the two colors separate along
component 1; and which years sit at the top of the plot on component 2.

{{< pagebreak >}}

## Problem Set Reflections (10 points)

Every problem set ends with these reflections. Answer them honestly. You are
graded on the specificity and honesty in the reflections you show, not on how
much AI you used or did not use. You cannot use AI for this part, and we are
not checking for spelling or grammar. Write as much as each answer needs.

Write your answer under each heading.

### AI use

List each substantive use of an AI assistant: the tool and model, what you
asked for, what it gave back, how you checked it, and whether you kept or
rejected the result. Write "None" if you used none.

*Your answer:*

### People consulted

Who you talked to about this assignment, and what about.

*Your answer:*

### Where you got stuck

The step that blocked you, and what got you unstuck.

*Your answer:*

### What you understand now

One thing you understand now that you did not when you started.

*Your answer:*

### Hours spent

An honest estimate of the time this took.

*Your answer:*

{{< pagebreak >}}

## Appendix: where the data came from

`data/speeches.csv` is the cleaned output of Problem Set 4's Section 1,
restricted to the addresses from 1934 on. The texts come from the Project
Gutenberg collection *Complete State of the Union Addresses*, extended by the
course with the addresses from 2002 on from the American Presidency Project and
whitehouse.gov, and the 2026 address from the American Presidency Project. The
party labels come from `presidents.csv`, the standard historical record of
presidential party affiliations. Whitespace has been collapsed to single
spaces, the curly apostrophe has been replaced by the straight one, and the
`decade` column is the year rounded down to a multiple of ten. Five of the 96
rows (1989, 1993, 2001, 2009, and 2025) are a new president's first address to
a joint session rather than a formal State of the Union; the file counts them,
and so does this problem set.
The transcripts of 2025 and 2026 keep stage directions and audience labels.

The table is supplied so that this problem set stands on its own. If you built
your own version in Problem Set 4, you may compare the two, but use the supplied
file for every result you report here so that your numbers and the answer key
refer to the same 96 rows.